In [ ]:
#EXERCICE 1

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from io import StringIO

# Chargement des données à partir du fichier texte fourni
data = pd.read_csv(StringIO(csv_content))

# Afficher les premières lignes
print("Aperçu des données :")
print(data.head())

# Informations sur le dataset
print("\nInformations sur les colonnes :")
print(data.info())

# Statistiques descriptives
print("\nStatistiques descriptives :")
print(data.describe())

# Compter les cas positifs et négatifs (diabète = 1 pour positif, 0 pour négatif)
diabetes_counts = data['diabetes'].value_counts()
print(f"\nNombre de cas négatifs (diabète = 0) : {diabetes_counts[0]}")
print(f"Nombre de cas positifs (diabète = 1) : {diabetes_counts[1]}")
print(f"Total : {len(data)}")

# Division en ensembles d'entraînement et de test (80% train, 20% test)
X = data.drop('diabetes', axis=1)
y = data['diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTaille de l'ensemble d'entraînement : {len(X_train)}")
print(f"Taille de l'ensemble de test : {len(X_test)}")

Aperçu des données :
Les données contiennent 9 colonnes : gender, age, hypertension, heart_disease, smoking_history, bmi, HbA1c_level, blood_glucose_level, diabetes.

Nombre de cas :
Cas négatifs (diabète = 0) : 91,570

Cas positifs (diabète = 1) : 8,430

Total : 100,000 patients

Le jeu de données est déséquilibré : seulement 8.43% des patients ont du diabète.

Division des données :
Ensemble d'entraînement : 80,000 patients (80%)

Ensemble de test : 20,000 patients (20%)

Les ensembles sont stratifiés par la variable cible pour maintenir la proportion de cas diabétiques dans les deux ensembles (stratify=y).

##EXERCICE 2

our ce problème de prédiction du diabète, plusieurs modèles de classification peuvent être envisagés :

Modèles recommandés :

Régression Logistique - Idéal pour commencer car :

Interprétable (on peut voir l'importance de chaque facteur de risque)

Rapide à entraîner

Fonctionne bien avec des relations linéaires

Donne des probabilités de risque

Random Forest - Très pertinent ici car :

Gère bien les relations non linéaires

Robuste face aux outliers

Peut gérer des données déséquilibrées

Donne l'importance des features (ex: quel facteur prédit le mieux le diabète)

XGBoost - Performant pour ce type de problème car :

Très performant sur les données tabulaires

Gère bien le déséquilibre des classes

Offre de bons paramètres de régularisation

KNN (à tester) - Simple mais peut être efficace

Pourquoi ces modèles ? Le diabète est influencé par des facteurs comme l'IMC, la glycémie, l'âge, etc. - des relations qui peuvent être capturées par ces modèles.

Faut-il normaliser les données ?
OUI, il faut normaliser les données car :

Les échelles sont très différentes :

age : 0-80 ans

bmi : 10-70

HbA1c_level : 3-9

blood_glucose_level : 80-300

Certains modèles sont sensibles aux échelles :

Régression Logistique (gradient descent)

KNN (distances)

SVM

PCA si utilisé

Modèles qui n'ont PAS besoin de normalisation :

Arbres de décision

Random Forest

XGBoost (mais peut aider parfois)

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Variables numériques à normaliser
numeric_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']

# Variables catégorielles à encoder
categorical_features = ['gender', 'smoking_history']

# Création du préprocesseur
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', LabelEncoder(), categorical_features)  # À adapter avec OneHotEncoder
])

# Appliquer sur les données d'entraînement et de test
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

In [ ]:
#EXERCICE 3

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from io import StringIO

# Chargement des données
data = pd.read_csv(StringIO(csv_content))

# Séparation des features et de la cible
X = data.drop('diabetes', axis=1)
y = data['diabetes']

# Division en ensembles d'entraînement et de test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille entraînement : {len(X_train)}")
print(f"Taille test : {len(X_test)}")
print(f"Proportion diabète train : {y_train.mean():.4f}")
print(f"Proportion diabète test : {y_test.mean():.4f}")

# Définition des colonnes
numeric_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
categorical_features = ['gender', 'smoking_history']
binary_features = ['hypertension', 'heart_disease']

# Création du préprocesseur
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('bin', 'passthrough', binary_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

# Création du pipeline avec régression logistique
# Augmentation de max_iter pour assurer la convergence
log_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])

# Entraînement du modèle
print("\nEntraînement du modèle de régression logistique...")
log_reg_pipeline.fit(X_train, y_train)

# Prédictions sur l'ensemble d'entraînement et de test
y_train_pred = log_reg_pipeline.predict(X_train)
y_test_pred = log_reg_pipeline.predict(X_test)

# Évaluation
print("\n" + "="*50)
print("RÉSULTATS DE LA RÉGRESSION LOGISTIQUE")
print("="*50)

print(f"\nAccuracy sur l'entraînement : {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Accuracy sur le test : {accuracy_score(y_test, y_test_pred):.4f}")

print("\nRapport de classification - Ensemble TEST :")
print(classification_report(y_test, y_test_pred, target_names=['Non diabétique', 'Diabétique']))

print("\nMatrice de confusion - Ensemble TEST :")
print(confusion_matrix(y_test, y_test_pred))

In [ ]:
#EXERCICE 4

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from io import StringIO

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Chargement des données
data = pd.read_csv(StringIO(csv_content))

# Préparation des données
X = data.drop('diabetes', axis=1)
y = data['diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Définition des colonnes
numeric_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
categorical_features = ['gender', 'smoking_history']
binary_features = ['hypertension', 'heart_disease']

# Pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('bin', 'passthrough', binary_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

log_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])

# Entraînement
log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)

# Calcul des métriques
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("="*60)
print("MÉTRIQUES D'ÉVALUATION - RÉGRESSION LOGISTIQUE")
print("="*60)
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

# FIGURE 1 : Score de précision (Accuracy)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Graphique 1 : Barre d'accuracy
ax1 = axes[0, 0]
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy, precision, recall, f1]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
bars = ax1.bar(metrics_names, metrics_values, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_ylim(0, 1)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Métriques de Performance du Modèle', fontsize=14, fontweight='bold')
ax1.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Seuil 0.8')
ax1.legend()
for bar, val in zip(bars, metrics_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Graphique 2 : Matrice de confusion
ax2 = axes[0, 1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2, 
            xticklabels=['Non Diabétique', 'Diabétique'],
            yticklabels=['Non Diabétique', 'Diabétique'])
ax2.set_xlabel('Prédiction', fontsize=12)
ax2.set_ylabel('Vérité terrain', fontsize=12)
ax2.set_title('Matrice de Confusion', fontsize=14, fontweight='bold')

# Ajout des pourcentages dans la matrice
total = np.sum(cm)
for i in range(2):
    for j in range(2):
        percentage = cm[i, j] / total * 100
        ax2.text(j+0.5, i+0.7, f'({percentage:.1f}%)', 
                ha='center', va='center', fontsize=10, color='gray')

# Graphique 3 : Précision, Rappel, F1-Score par classe
ax3 = axes[1, 0]
report = classification_report(y_test, y_pred, target_names=['Non Diabétique', 'Diabétique'], output_dict=True)
classes = ['Non Diabétique', 'Diabétique']
precision_scores = [report['Non Diabétique']['precision'], report['Diabétique']['precision']]
recall_scores = [report['Non Diabétique']['recall'], report['Diabétique']['recall']]
f1_scores = [report['Non Diabétique']['f1-score'], report['Diabétique']['f1-score']]

x = np.arange(len(classes))
width = 0.25

bars1 = ax3.bar(x - width, precision_scores, width, label='Précision', color='#2E86AB', edgecolor='black')
bars2 = ax3.bar(x, recall_scores, width, label='Rappel', color='#F18F01', edgecolor='black')
bars3 = ax3.bar(x + width, f1_scores, width, label='F1-Score', color='#A23B72', edgecolor='black')

ax3.set_ylabel('Score', fontsize=12)
ax3.set_title('Métriques par Classe', fontsize=14, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(classes)
ax3.legend(loc='lower right')
ax3.set_ylim(0, 1.1)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2, height + 0.01, 
                f'{height:.3f}', ha='center', va='bottom', fontsize=10)

# Graphique 4 : Analyse des erreurs (Types d'erreurs)
ax4 = axes[1, 1]
tn, fp, fn, tp = cm.ravel()
error_types = ['Vrais Négatifs', 'Faux Positifs', 'Faux Négatifs', 'Vrais Positifs']
error_counts = [tn, fp, fn, tp]
error_colors = ['#2ECC71', '#E74C3C', '#E74C3C', '#2ECC71']
bars = ax4.bar(error_types, error_counts, color=error_colors, edgecolor='black', linewidth=1.5)
ax4.set_ylabel('Nombre de patients', fontsize=12)
ax4.set_title('Analyse des Erreurs de Classification', fontsize=14, fontweight='bold')
ax4.tick_params(axis='x', rotation=15)

for bar, count in zip(bars, error_counts):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
            f'{count}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_metrics_diabetes.png', dpi=150, bbox_inches='tight')
plt.show()

# Graphique supplémentaire : Évolution des métriques seuil par seuil
fig2, ax5 = plt.subplots(figsize=(12, 6))

# Calcul des probabilités
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]
thresholds = np.linspace(0, 1, 50)
precisions = []
recalls = []
f1_scores_list = []

for thresh in thresholds:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    precisions.append(precision_score(y_test, y_pred_thresh, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_thresh, zero_division=0))
    f1_scores_list.append(f1_score(y_test, y_pred_thresh, zero_division=0))

ax5.plot(thresholds, precisions, 'b-', label='Précision', linewidth=2)
ax5.plot(thresholds, recalls, 'r-', label='Rappel', linewidth=2)
ax5.plot(thresholds, f1_scores_list, 'g-', label='F1-Score', linewidth=2)
ax5.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Seuil par défaut (0.5)')
ax5.set_xlabel('Seuil de décision', fontsize=12)
ax5.set_ylabel('Score', fontsize=12)
ax5.set_title('Impact du Seuil de Décision sur les Métriques', fontsize=14, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Trouver le seuil optimal (max F1)
optimal_idx = np.argmax(f1_scores_list)
optimal_threshold = thresholds[optimal_idx]
ax5.plot(optimal_threshold, f1_scores_list[optimal_idx], 'go', markersize=10, 
         label=f'Seuil optimal F1 = {optimal_threshold:.3f}')
ax5.legend()

plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("ANALYSE DES RÉSULTATS")
print("="*60)

print("\n **1. SCORE DE PRÉCISION (ACCURACY) :**")
print(f"   Accuracy = {accuracy:.2%}")
print("    Interprétation : Le modèle est correct dans 95% des cas.")
print("     Attention : Cette métrique est trompeuse car la classe majoritaire (non diabétique) domine.")
print("      95% d'accuracy pourrait être obtenu en prédisant 'non diabétique' pour tout le monde.")

print("\n **2. MATRICE DE CONFUSION :**")
print(f"   • Vrais Négatifs (TN) : {tn:,} patients non diabétiques correctement identifiés")
print(f"   • Vrais Positifs (TP) : {tp:,} patients diabétiques correctement identifiés")
print(f"   • Faux Positifs (FP) : {fp:,} patients sains classés à tort comme diabétiques")
print(f"   • Faux Négatifs (FN) : {fn:,} patients diabétiques non détectés")
print("\n    Points positifs : Très peu de faux positifs (seulement 0.3% des patients)")
print("    Problème majeur : 930 faux négatifs ! Soit 55% des vrais diabétiques non détectés.")
print("      C'est inacceptable dans un contexte médical.")

print("\n **3. MÉTRIQUES PAR CLASSE :**")
print("\n   Classe NON DIABÉTIQUE (majoritaire) :")
print(f"   • Précision : {report['Non Diabétique']['precision']:.3f} - Quand on dit 'non diabétique', on a raison dans 95% des cas")
print(f"   • Rappel    : {report['Non Diabétique']['recall']:.3f} - On détecte 99.7% des vrais non-diabétiques")
print(f"   • F1-Score  : {report['Non Diabétique']['f1-score']:.3f} - Excellent")

print("\n   Classe DIABÉTIQUE (minoritaire) :")
print(f"   • Précision : {report['Diabétique']['precision']:.3f} - 87% des prédictions 'diabétique' sont correctes")
print(f"   • Rappel    : {report['Diabétique']['recall']:.3f} - On ne détecte que 45% des vrais diabétiques !!!")
print(f"   • F1-Score  : {report['Diabétique']['f1-score']:.3f} - Médiocre à cause du faible rappel")

print("\n **4. ANALYSE SEUIL PAR SEUIL :**")
print(f"   • Seuil par défaut (0.5) : Recall = {recall:.3f}")
print(f"   • Seuil optimal pour F1 : {optimal_threshold:.3f}")
print("   • Si on abaisse le seuil à 0.3 :")
print("     - Le rappel augmenterait (on détecterait plus de diabétiques)")
print("     - Mais la précision diminuerait (plus de faux positifs)")
print("     - À calibrer selon le coût médical des erreurs")

print("\n **RECOMMANDATIONS :**")
print("   1. Ne pas se fier uniquement à l'accuracy (95% semble bon mais cache un problème)")
print("   2. Prioriser le RAPPEL pour la classe diabétique (on veut éviter les faux négatifs)")
print("   3. Essayer d'autres modèles : Random Forest, XGBoost")
print("   4. Utiliser SMOTE pour rééquilibrer les classes")
print("   5. Ajuster le seuil de décision selon le contexte médical")

In [ ]:
#EXERCICE 5

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

# Chargement des données
data = pd.read_csv(StringIO(csv_content))

# Préparation des données
X = data.drop('diabetes', axis=1)
y = data['diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Définition des colonnes
numeric_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
categorical_features = ['gender', 'smoking_history']
binary_features = ['hypertension', 'heart_disease']

# Pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('bin', 'passthrough', binary_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

log_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])

# Entraînement
log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

# Obtenir les données transformées
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Calcul des métriques pour différents seuils
thresholds = np.linspace(0, 1, 50)
precisions = []
recalls = []
f1_scores = []

for thresh in thresholds:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    precisions.append(precision_score(y_test, y_pred_thresh, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_thresh, zero_division=0))
    f1_scores.append(f1_score(y_test, y_pred_thresh, zero_division=0))

optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print("="*70)
print("VISUALISATION DES FRONTIÈRES DE DÉCISION")
print("="*70)
print(f"Seuil par défaut : 0.5 | Seuil optimal (F1) : {optimal_threshold:.3f}")

# Création des graphiques
fig = plt.figure(figsize=(18, 12))

# 1. Visualisation avec PCA (2 dimensions)
ax1 = fig.add_subplot(2, 3, 1)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_test_transformed)

# Création d'une grille pour la frontière de décision
x_min, x_max = X_pca[:, 0].min() - 0.5, X_pca[:, 0].max() + 0.5
y_min, y_max = X_pca[:, 1].min() - 0.5, X_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))

# Prédiction sur la grille (nécessite un modèle entraîné sur les composantes PCA)
from sklearn.linear_model import LogisticRegression
pca_model = LogisticRegression(random_state=42, class_weight='balanced')
pca_model.fit(X_pca, y_test)
Z = pca_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Affichage de la frontière
ax1.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
scatter = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y_test, cmap='RdYlBu', 
                      edgecolor='black', alpha=0.7, s=30)
ax1.set_xlabel('PC1', fontsize=11)
ax1.set_ylabel('PC2', fontsize=11)
ax1.set_title('Frontière de décision (PCA 2D)\nSeuil = 0.5', fontsize=12, fontweight='bold')
ax1.legend(*scatter.legend_elements(), title="Classe réelle", labels=['Non diabétique', 'Diabétique'])
ax1.grid(True, alpha=0.3)

# 2. Impact du seuil sur la frontière
ax2 = fig.add_subplot(2, 3, 2)
for thresh in [0.3, 0.5, 0.7]:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    precision = precision_score(y_test, y_pred_thresh, zero_division=0)
    recall = recall_score(y_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)
    marker = 'o' if thresh == 0.5 else 's' if thresh == 0.3 else '^'
    size = 150 if thresh == 0.5 else 100
    ax2.scatter(recall, precision, s=size, marker=marker, 
                label=f'Seuil={thresh} (F1={f1:.3f})', alpha=0.8)
    ax2.annotate(f'  seuil={thresh}', (recall, precision), fontsize=9)

# Courbe PR
ax2.plot(recalls, precisions, 'b-', alpha=0.5, linewidth=2, label='Courbe PR')
ax2.scatter(recalls[optimal_idx], precisions[optimal_idx], s=200, c='red', 
            marker='*', label=f'Seuil optimal (F1={f1_scores[optimal_idx]:.3f})', zorder=5)
ax2.set_xlabel('Rappel (Recall)', fontsize=11)
ax2.set_ylabel('Précision (Precision)', fontsize=11)
ax2.set_title('Courbe Précision-Rappel\nImpact du seuil de décision', fontsize=12, fontweight='bold')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

# 3. Probabilités prédites par classe
ax3 = fig.add_subplot(2, 3, 3)
non_diabetic_proba = y_proba[y_test == 0]
diabetic_proba = y_proba[y_test == 1]

ax3.hist(non_diabetic_proba, bins=30, alpha=0.7, label='Non diabétique', color='blue', density=True)
ax3.hist(diabetic_proba, bins=30, alpha=0.7, label='Diabétique', color='red', density=True)
ax3.axvline(x=0.5, color='gray', linestyle='--', linewidth=2, label='Seuil par défaut')
ax3.axvline(x=optimal_threshold, color='green', linestyle='--', linewidth=2, label=f'Seuil optimal ({optimal_threshold:.2f})')
ax3.set_xlabel('Probabilité prédite du diabète', fontsize=11)
ax3.set_ylabel('Densité', fontsize=11)
ax3.set_title('Distribution des probabilités prédites\npar classe réelle', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Frontière pour les 2 features les plus importantes
ax4 = fig.add_subplot(2, 3, 4)
# Sélection des 2 features les plus importantes selon les coefficients du modèle
feature_names = (numeric_features + binary_features + 
                 [f'smoking_history_{cat}' for cat in ['current', 'ever', 'former', 'not current'] if cat != 'No Info'] + 
                 ['gender_Male'])
coefficients = log_reg_pipeline.named_steps['classifier'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': np.abs(coefficients)})
top_features = coef_df.nlargest(2, 'coefficient')['feature'].values

print(f"\nTop 2 features les plus importantes : {top_features}")

# Pour la visualisation, on utilise les données originales avec ces 2 features
feature1_idx = numeric_features.index(top_features[0]) if top_features[0] in numeric_features else None
feature2_idx = numeric_features.index(top_features[1]) if top_features[1] in numeric_features else None

if feature1_idx is not None and feature2_idx is not None:
    X_test_num = X_test[numeric_features].values
    x_feat = X_test_num[:, feature1_idx]
    y_feat = X_test_num[:, feature2_idx]
    
    # Création de la grille
    x_min, x_max = x_feat.min() - 0.5, x_feat.max() + 0.5
    y_min, y_max = y_feat.min() - 0.5, y_feat.max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    
    # Prédiction sur la grille (simplifiée)
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    grid_predictions = []
    for point in grid_points:
        temp_df = X_test.iloc[0:1].copy()
        temp_df[top_features[0]] = point[0]
        temp_df[top_features[1]] = point[1]
        pred = log_reg_pipeline.predict_proba(temp_df)[0, 1]
        grid_predictions.append(pred)
    Z_grid = np.array(grid_predictions).reshape(xx.shape)
    
    ax4.contourf(xx, yy, Z_grid, levels=20, cmap='RdYlBu', alpha=0.6)
    ax4.contour(xx, yy, Z_grid, levels=[0.5], colors='black', linewidths=2, label='Frontière seuil=0.5')
    ax4.contour(xx, yy, Z_grid, levels=[optimal_threshold], colors='green', linewidths=2, 
                linestyles='--', label=f'Frontière seuil={optimal_threshold:.2f}')
    scatter = ax4.scatter(x_feat, y_feat, c=y_test, cmap='RdYlBu', edgecolor='black', alpha=0.7, s=30)
    ax4.set_xlabel(top_features[0], fontsize=11)
    ax4.set_ylabel(top_features[1], fontsize=11)
    ax4.set_title(f'Frontière de décision\n{top_features[0]} vs {top_features[1]}', fontsize=12, fontweight='bold')
    ax4.legend(['Seuil 0.5', f'Seuil {optimal_threshold:.2f}'], loc='best')
    ax4.grid(True, alpha=0.3)

# 5. Heatmap de la matrice de confusion normalisée
ax5 = fig.add_subplot(2, 3, 5)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

im = ax5.imshow(cm_normalized, cmap='Blues', interpolation='nearest')
ax5.set_xticks([0, 1])
ax5.set_yticks([0, 1])
ax5.set_xticklabels(['Non diabétique', 'Diabétique'])
ax5.set_yticklabels(['Non diabétique', 'Diabétique'])
ax5.set_xlabel('Prédiction', fontsize=11)
ax5.set_ylabel('Réel', fontsize=11)
ax5.set_title('Matrice de confusion normalisée\n(par ligne)', fontsize=12, fontweight='bold')

for i in range(2):
    for j in range(2):
        text = ax5.text(j, i, f'{cm_normalized[i, j]:.2%}\n({cm[i, j]})',
                       ha="center", va="center", color="white" if cm_normalized[i, j] > 0.5 else "black", fontsize=10)
plt.colorbar(im, ax=ax5)

# 6. Métriques par seuil
ax6 = fig.add_subplot(2, 3, 6)
ax6.plot(thresholds, precisions, 'b-', label='Précision', linewidth=2)
ax6.plot(thresholds, recalls, 'r-', label='Rappel', linewidth=2)
ax6.plot(thresholds, f1_scores, 'g-', label='F1-Score', linewidth=2)
ax6.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Seuil par défaut (0.5)')
ax6.axvline(x=optimal_threshold, color='green', linestyle='--', linewidth=2, 
            label=f'Seuil optimal ({optimal_threshold:.3f})')
ax6.fill_between(thresholds, 0, 1, where=(thresholds >= 0.4) & (thresholds <= 0.6), 
                 alpha=0.2, color='yellow', label='Zone seuil standard')
ax6.set_xlabel('Seuil de décision', fontsize=11)
ax6.set_ylabel('Score', fontsize=11)
ax6.set_title('Évolution des métriques\nselon le seuil de décision', fontsize=12, fontweight='bold')
ax6.legend(loc='center right')
ax6.grid(True, alpha=0.3)
ax6.set_xlim(0, 1)
ax6.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('decision_boundary_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

# Graphique supplémentaire : Visualisation 3D des probabilités
fig2 = plt.figure(figsize=(14, 8))
ax7 = fig2.add_subplot(111, projection='3d')

# Utilisation des 2 features les plus importantes pour la 3D
if feature1_idx is not None and feature2_idx is not None:
    x_3d = X_test_num[:, feature1_idx]
    y_3d = X_test_num[:, feature2_idx]
    z_3d = y_proba
    
    scatter = ax7.scatter(x_3d, y_3d, z_3d, c=y_test, cmap='RdYlBu', s=30, alpha=0.7)
    ax7.set_xlabel(top_features[0], fontsize=10)
    ax7.set_ylabel(top_features[1], fontsize=10)
    ax7.set_zlabel('Probabilité diabète', fontsize=10)
    ax7.set_title('Visualisation 3D des probabilités prédites\n' + 
                  f'({top_features[0]} vs {top_features[1]})', fontsize=12, fontweight='bold')
    
    # Plan de seuil
    xx_3d, yy_3d = np.meshgrid(np.linspace(x_3d.min(), x_3d.max(), 20),
                               np.linspace(y_3d.min(), y_3d.max(), 20))
    zz_3d = np.full_like(xx_3d, 0.5)
    ax7.plot_surface(xx_3d, yy_3d, zz_3d, alpha=0.3, color='gray')
    
    plt.colorbar(scatter, ax=ax7, label='Probabilité')
    
plt.tight_layout()
plt.savefig('probability_3d_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

# Analyse supplémentaire
print("\n" + "="*70)
print("ANALYSE DE LA FRONTIÈRE DE DÉCISION")
print("="*70)

print("\n **1. FRONTIÈRE DE DÉCISION (PCA) :**")
print("   • La frontière linéaire sépare partiellement les deux classes")
print("   • Beaucoup de chevauchement entre les classes : des diabétiques et non-diabétiques")
print("     se trouvent dans la même région de l'espace PCA")
print("   • Explique pourquoi le recall est faible : le modèle est prudent")

print("\n **2. IMPACT DU SEUIL :**")
print(f"   • Seuil bas (0.3)  : Plus de sensibilité (recall élevé) mais moins de précision")
print(f"   • Seuil haut (0.7) : Plus de précision mais moins de recall")
print(f"   • Seuil optimal : {optimal_threshold:.3f} (maximise le F1-score = {f1_scores[optimal_idx]:.3f})")

print("\n **3. DISTRIBUTION DES PROBABILITÉS :**")
print("   • Les non-diabétiques ont majoritairement des probabilités faibles (< 0.2)")
print("   • Les diabétiques ont des probabilités plus étalées (0.2 à 0.8)")
print("   • Fort chevauchement entre 0.2 et 0.6 : zone d'incertitude")

print("\n **4. FRONTIÈRE POUR LES 2 FEATURES PRINCIPALES :**")
print(f"   • Features les plus importantes : {top_features[0]} et {top_features[1]}")
print("   • La frontière n'est pas parfaitement linéaire dans cet espace")
print("   • Zone de transition importante où les classes se mélangent")

print("\n **RECOMMANDATIONS POUR L'AMÉLIORATION :**")
print("   1. Utiliser un modèle non linéaire (Random Forest, SVM avec RBF)")
print("   2. Créer des features d'interaction (ex: glycémie × HbA1c)")
print("   3. Ajuster le seuil selon le contexte :")
print("      - Dépistage : utiliser seuil bas (0.3-0.4) pour maximiser la détection")
print("      - Diagnostic : utiliser seuil haut (0.6-0.7) pour éviter les faux positifs")
print("   4. Envisager un modèle à 2 étages pour les cas ambigus (probabilité 0.3-0.6)")

In [ ]:
#EXERCICE 6

# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from io import StringIO

# --- 1. Chargement et préparation des données (reprise de l'exercice 3) ---
# Supposons que 'csv_content' contient le contenu du fichier
data = pd.read_csv(StringIO(csv_content))

X = data.drop('diabetes', axis=1)
y = data['diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Définition des colonnes
numeric_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
categorical_features = ['gender', 'smoking_history']
binary_features = ['hypertension', 'heart_disease']

# Pipeline de prétraitement et du modèle
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('bin', 'passthrough', binary_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

log_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])

# Entraînement du modèle
log_reg_pipeline.fit(X_train, y_train)

# --- 2. Prédiction des probabilités sur l'ensemble de test ---
# La fonction predict_proba retourne les probabilités pour les deux classes [0, 1].
# On sélectionne la probabilité de la classe positive (diabète = 1).
y_pred_proba = log_reg_pipeline.predict_proba(X_test)[::, 1]

# --- 3. Calcul des métriques pour la courbe ROC ---
# fpr: False Positive Rate (Taux de faux positifs = 1 - Spécificité)
# tpr: True Positive Rate (Taux de vrais positifs = Sensibilité)
# thresholds: les différents seuils de décision utilisés pour le calcul
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred_proba)

# --- 4. Calcul de l'AUC (Area Under the Curve) ---
# L'AUC mesure la performance globale du modèle, quelle que soit la classe.
# Une valeur de 1.0 est parfaite, 0.5 est aléatoire.
auc = metrics.roc_auc_score(y_test, y_pred_proba)

# --- 5. Tracé de la courbe ROC ---
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Régression Logistique (AUC = {auc:.3f})', color='darkorange', linewidth=2)

# Tracé de la ligne de référence (classificateur aléatoire)
plt.plot([0, 1], [0, 1], 'k--', label='Classificateur aléatoire (AUC = 0.5)')

# Personnalisation du graphique
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (1 - Spécificité)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (Sensibilité)', fontsize=12)
plt.title('Courbe ROC pour la Prédiction du Diabète', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()